# Dataset and Dataloader to train a NN : introduction to batching

**Dataset classic approach**:

- classic split of dataset in training and validation sets.
- model's training + evaluation on the whole dataset.

**DataLoader approach == Batching**:

- split dataset in training and validation sets by indexes with ***Subsets** (saves memory because avoids to copy data) [recommanded]*.
- employ Dataloader to split the training and validation sets in batches.
- model's training on batches.
- model's evaluation on batches.

> *DataLoader* is a command splitting the original (training and validation) dataset into batch. It returns an iterable object over the batches. Then, the training and the evaluation phases of the model are performed on batches, instaed than on the whole dataset.

> **Why to use Dataloader (i.e. batching) instaed than a classic dataset?**: *Training and evaluation are performed on small batches, instaed than the whole dataset, which lowers the computational cost.* With real data, the DataLoader approach is usually preferred.

> **PROS**:
> 1) Less memory required.
> 2) Gradually processing of data.
> 3) Processes can be parallelized, i.e. GPUs can be used.
     Rmk) If $n$ GPUs are avaiable, set num_workers at least $n/2$, to achieve fast results.
> 4) Faster training and evaluation.
> 5) Randomness is introduced, because data are shuffled to create batches. 

> **CONS**
> 1) Too small batches make training noiser, because estimations on too small datasets can vary considerably one from another.
> 2) Too large batches are problematic, because GPUs have a limited amount of memory and you might run out of GPUs memory. Furthermore, the complexity might equal that of a classic training.

> **What is an adequate batch size?** A tradeoff between too small and too large. It's chosen as an hyperparameter before training the model.


In [2]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
import torch
import torchinfo

from torchvision.datasets import MNIST

# import dataset, dataloader, subset
from torch.utils.data import Dataset, DataLoader, Subset

## Dataset

In [4]:
class MNISTDataset(Dataset):
    
    def __init__(self):
        
        data = MNIST('mnist', download = True)
        
        self.images, self.labels = data.data, data.targets
        
    def __len__(self):                   # length of dataset
        
        return len(self.images)
    
    def __getitem__(self, idx : int):    # element of index idx
        
        image = self.images[idx].float().view(28*28)/255.0
        
        label = self.labels[idx]
        
        return image, label
        

In [5]:
dataset = MNISTDataset()

In [6]:
# PROCEED AS IN PRECEDENT NOTEBOOKS

# - split dataset into training and validation part
# - call a model
# - call a loss
# - call a optimizer
# - train and evaluate the model, on 10000 epochs

## DataLoader

**Dataloader**: iterates over a dataset and returns batches

- splits the dataset in batches

- returns an iterable object over batches, i.e. an object you can only go through batches

**Training and evaluation with DataLoader, i.e. on batches**

*Fewer epochs are needed*

-> *The training phase*:

- Loop over batches from the dataloader_training.
- Train the model on each batch.

-> *The evaluation phase*:

- Loop over batches from the dataloader_validation.
- Prediction on each batch.
- Extract target value on each batch. 


In [ ]:
# PARTITION INDEXES BETWEEN TRAINING AND VALIDATION

# This approach saves memory, because allows for not copying data

idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)

# define TRAINING SET
train_dataset = Subset(dataset, idx_train)  # train_dataset[i] = dataset[idx_train[i]]

# define VALIDATION SET
valid_dataset = Subset(dataset, idx_valid)  # valid_dataset[i] = dataset[idx_valid[i]]


In [ ]:
# OPTIONAL ARGUMENT,
#
# TELLING HOW TO FORM THE BATCH

def collate(batch):

    images = []                             # list of tensors
    labels = []                             # list of tensors

    for image, label in batch:
        images.append(image)
        labels.append(label)

    images = torch.stack(images, dim=0)     # make it a tensor
    labels = torch.tensor(labels)           # make it a tensor

    return images, labels                   # return images and labels as tensors


In [ ]:
# DEFINE DATALOADER FOR TRAINING, BATCHING : iteratable object of batches

train_dataloader = DataLoader(

    train_dataset,            # MUST BE SET

    batch_size = 32,          # MUST BE SET
    
    shuffle = True,           # change order in batch at each epoch

    num_workers = 0,          # number of parallel processes to set if training on GPU
                              # = 0 on CPU

    drop_last = True,         # if the batch size does not divide exactly the dataset size,
                              # this command deletes the remaining part

    collate_fn = collate      # OPTIONAL, tells how to form the batch
                              # default: stack data as Torch Tensors
)

In [16]:
# EXAMPLE: READING FIRST ELEMENT OF DATALOADER
# 
# extract the first batch
# from train_dataloader which is an iterable object

for batch in train_dataloader:
    break

#OR
#batch = next( iter(train_dataloader) ) 

print("Number of batches:", len(train_dataloader))      # number of batches
print("Images shape:", batch[0].shape)                  # images
print("Labels shape:", batch[1].shape)                  # labels


Number of batches: 1687
Images shape: torch.Size([32, 784])
Labels shape: torch.Size([32])


In [ ]:
# DEFINE DATALOADER FOR VALIDATION, BATCHING : iteratable object of batches

valid_dataloader = DataLoader(
    valid_dataset,

    batch_size = 32,

    shuffle = False,

    num_workers = 0,

    drop_last = True,
    
    collate_fn = collate
)

### Example of training on a DataLoader

In [ ]:
# DEFINE MODEL MLP

class TorchMLP(torch.nn.Module):
    
    def __init__(self, input_dim, hidden_dim, num_classes):
        
        super().__init__()
        
        all_dims = [input_dim] + list(hidden_dim) + [num_classes]
        
        layers = []
        
        for dim1, dim2 in zip(all_dims[:-1], all_dims[1:]):
            
            layers.append(torch.nn.Linear(dim1, dim2))
            
            layers.append(torch.nn.ReLU())

        layers = layers[:-1]

        self.model = torch.nn.Sequential(*layers)

        
    def forward(self, x):
        
        return self.model(x)
    

In [13]:
# CALL MODEL MLP

model_MLP = TorchMLP(784, [100, 200, 100], 10)

loss_MLP = torch.nn.CrossEntropyLoss()

optimizer_MLP = torch.optim.Adam(model_MLP.parameters(), lr = 0.002)

In [14]:

torchinfo.summary(model_MLP)

Layer (type:depth-idx)                   Param #
TorchMLP                                 --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       78,500
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       20,200
│    └─ReLU: 2-4                         --
│    └─Linear: 2-5                       20,100
│    └─ReLU: 2-6                         --
│    └─Linear: 2-7                       1,010
Total params: 119,810
Trainable params: 119,810
Non-trainable params: 0

In [ ]:
# TRAINING AND EVALUATION WITH BATCHING


for epoch in range(10):
  
  # TRAINING ON EACH BATCH
  for images, label in train_dataloader:      # training on each batch
    
    optimizer_MLP.zero_grad()                                     
    y_pred = model_MLP(images)                # prediction on batch   
    loss_result = loss_MLP(y_pred, label)     # loss on batch
    loss_result.backward()                                
    optimizer_MLP.step()       


  # EVALUATION ON EACH BATCH
  all_preds = []                # collect prediction on batches
  all_labels = []               # collect labels on batches
  
  with torch.no_grad():                                   # delete gradients, to save memory

      for images, labels in valid_dataloader:             # evaluation on each batch        

         logits = model_MLP(images)                       # prediction on each batch

         all_preds.extend(torch.argmax(logits, dim=1)) 

         all_labels.extend(labels)      
                                                          
  print(accuracy_score(all_preds, all_labels))

0.9598930481283422
0.9613970588235294
0.9712566844919787
0.9685828877005348
0.9697526737967914
0.9730949197860963
0.9762700534759359
0.9715909090909091
0.9747660427807486
0.9724264705882353
